In [1]:
import os
import re
import numpy as np
import sympy as sp
import openturns as ot
import matplotlib.pyplot as plt
import trimesh as tr

from math import pi
from joblib import Parallel, delayed

from scipy.optimize import minimize, basinhopping, \
                           differential_evolution, brute, shgo, check_grad, \
                           approx_fprime, fsolve, NonlinearConstraint, Bounds, approx_fprime

import otaf

from gldpy import GLD

ot.Log.Show(ot.Log.NONE)
np.set_printoptions(suppress=True)
ar = np.array

import logging
logging.basicConfig()
logging.getLogger().setLevel(logging.DEBUG)

# Solving issues regarding the handling of cylinders. 

Cylinders are kinda a pain in the ass to handle, as such here a very simple example where we have two concentric cylinders, one within each other, the outer one fixed, the inner one has translational and rotational play. So the parts are just one feature each.

In [2]:
# Some basic values:
height = 50
RAD1 = 20
RAD2 = 22

R0 = ar([[1, 0, 0], [0, 1, 0], [0, 0, 1]])
x_, y_, z_ = R0[0], R0[1], R0[2]

Frame1 = R0 #For now we can simply align them. Cylinders are oriented such that their axis is on x, the direction doesn't matter.
Frame2 = Frame1 #ar([-x_,y_,-z_]) #If they are the same there is no issue. Let's invert them. 

In [3]:
system_data = {
    "PARTS" : {
        '0' : {
            "a" : {
                "FRAME": Frame1, # Frame doesn't really matter, as long as x is aligned on the axis
                "ORIGIN": ar([0,0,0]), 
                "TYPE": "cylinder",
                "RADIUS": RAD2,
                "EXTENT_LOCAL": {"x_max": height/2, "x_min": -height/2},
                "INTERACTIONS": [f"P1a"], 
                "SURFACE_DIRECTION": "centripetal", #This feature is a hole
                "CONSTRAINTS_D": ["PERFECT"], # No defects, does not move     
            }
        },
        '1' : {
            "a" : {
                "FRAME": Frame2, # Frame doesn't really matter, as long as x is aligned on the axis
                "ORIGIN": ar([2,0,0]), 
                "TYPE": "cylinder",
                "RADIUS": RAD1,
                "EXTENT_LOCAL": {"x_max": height/2, "x_min": -height/2},
                "INTERACTIONS": [f"P0a"], 
                "SURFACE_DIRECTION": "centrifugal", #This feature is a cylinder        
            }
        }  
    },
    "LOOPS": {
        "COMPATIBILITY": {"LO" : "P0aA0 -> P1aA0" },
    },
    "GLOBAL_CONSTRAINTS": "3D",
}

In [4]:
system_data

{'PARTS': {'0': {'a': {'FRAME': array([[1, 0, 0],
           [0, 1, 0],
           [0, 0, 1]]),
    'ORIGIN': array([0, 0, 0]),
    'TYPE': 'cylinder',
    'RADIUS': 22,
    'EXTENT_LOCAL': {'x_max': 25.0, 'x_min': -25.0},
    'INTERACTIONS': ['P1a'],
    'SURFACE_DIRECTION': 'centripetal',
    'CONSTRAINTS_D': ['PERFECT']}},
  '1': {'a': {'FRAME': array([[1, 0, 0],
           [0, 1, 0],
           [0, 0, 1]]),
    'ORIGIN': array([2, 0, 0]),
    'TYPE': 'cylinder',
    'RADIUS': 20,
    'EXTENT_LOCAL': {'x_max': 25.0, 'x_min': -25.0},
    'INTERACTIONS': ['P0a'],
    'SURFACE_DIRECTION': 'centrifugal'}}},
 'LOOPS': {'COMPATIBILITY': {'LO': 'P0aA0 -> P1aA0'}},
 'GLOBAL_CONSTRAINTS': '3D'}

In [5]:
SDA = otaf.AssemblyDataProcessor(system_data)
SDA.generate_expanded_loops()
SDA.compatibility_loops_expanded

INFO:root:Validating system data dictionary structure.


{'LO': 'D0a -> GP0aA0P1aA0 -> Di1a'}

In [6]:
CLH = otaf.CompatibilityLoopHandling(SDA)
compatibility_expressions = CLH.get_compatibility_expression_from_FO_matrices()
compatibility_expressions

INFO:root:Initializing CompatibilityLoopHandling object.
INFO:root:Generating loop ID to matrix list dictionary
INFO:root:Initializing DeviationMatrix with index: 0, translations: , rotations: , inverse: False
INFO:root:Initializing basis variables.
DEBUG:root:Initialized 0 variables for the deviation matrix.
INFO:root:[Type: 'T', ID: -1] Initializing TransformationMatrix.
DEBUG:root:[Type: 'T', ID: -1] Calculating change of basis matrix.
DEBUG:root:[Type: 'T', ID: -1] Change of basis matrix calculated.
DEBUG:root:[Type: 'T', ID: -1] Calculating change of basis matrix.
DEBUG:root:[Type: 'T', ID: -1] Change of basis matrix calculated.
INFO:root:[Type: 'T', ID: -1] Generating transformation matrix.
DEBUG:root:[Type: 'T', ID: -1] Calculating change of basis matrix.
DEBUG:root:[Type: 'T', ID: -1] Change of basis matrix calculated.
INFO:root:[Type: 'T', ID: 0] Initializing TransformationMatrix.
INFO:root:[Type: 'G', ID: 0] Initializing GapMatrix with blocked translations: , blocked rotation

[gamma_d_1 - gamma_g_0,
 -beta_d_1 + beta_g_0,
 -alpha_g_0,
 u_g_0,
 -v_d_1 + v_g_0,
 -w_d_1 + w_g_0]

In [7]:
ILH = otaf.InterfaceLoopHandling(SDA, CLH, circle_resolution=3)
interface_constraints = [expr.evalf(9) for expr in ILH.get_interface_loop_expressions()]
interface_constraints

DEBUG:root:Processing cylinder-cylinder interaction for part 0 / surface a and part 1 / surface a.
DEBUG:root:Cylinder revolution axes are not facing.
DEBUG:root:Start cylinder to end cylinder origin vector in local [2. 0. 0.] and global [2. 0. 0.] coordinates
DEBUG:root:First cylinder local extent : -25.0 / 25.0
DEBUG:root:Second cylinder local extent : -25.0 / 25.0
DEBUG:root:	IF STATEMENT 2, NOT FACING
DEBUG:root:	IF STATEMENT 4, NOT FACING
DEBUG:root:Processing cylinder-cylinder interaction for part 1 / surface a and part 0 / surface a.
DEBUG:root:Cylinder revolution axes are not facing.
DEBUG:root:Start cylinder to end cylinder origin vector in local [-2.  0.  0.] and global [-2.  0.  0.] coordinates
DEBUG:root:First cylinder local extent : -25.0 / 25.0
DEBUG:root:Second cylinder local extent : -25.0 / 25.0
INFO:root:[Type: 'T', ID: -1] Initializing TransformationMatrix.
DEBUG:root:[Type: 'T', ID: -1] Calculating change of basis matrix.
DEBUG:root:[Type: 'T', ID: -1] Change of bas

[-23.0*gamma_g_0 - 1.0*v_g_0 + 2.0,
 19.9185843*beta_g_0 + 11.5*gamma_g_0 + 0.5*v_g_0 - 0.866025404*w_g_0 + 2.0,
 -19.9185843*beta_g_0 + 11.5*gamma_g_0 + 0.5*v_g_0 + 0.866025404*w_g_0 + 2.0,
 25.0*gamma_g_0 - 1.0*v_g_0 + 2.0,
 -21.6506351*beta_g_0 - 12.5*gamma_g_0 + 0.5*v_g_0 - 0.866025404*w_g_0 + 2.0,
 21.6506351*beta_g_0 - 12.5*gamma_g_0 + 0.5*v_g_0 + 0.866025404*w_g_0 + 2.0]

In [8]:
SOCAM = otaf.SystemOfConstraintsAssemblyModel(
    compatibility_expressions, interface_constraints
)

SOCAM.embedOptimizationVariable()

print(len(SOCAM.deviation_symbols), SOCAM.deviation_symbols)

INFO:root:[Type: SystemOfConstraintsAssemblyModel] Initializing SystemOfConstraintsAssemblyModel with 6 compatibility equations and 6 interface equations.
INFO:root:[Type: SystemOfConstraintsAssemblyModel] Retrieving free gap and deviation variables.
INFO:root:[Type: SystemOfConstraintsAssemblyModel] Free gap and deviation variables retrieved successfully.
INFO:root:[Type: SystemOfConstraintsAssemblyModel] Derived deviation symbols: [v_d_1, w_d_1, beta_d_1, gamma_d_1], gap symbols: [u_g_0, v_g_0, w_g_0, alpha_g_0, beta_g_0, gamma_g_0]
INFO:root:[Type: SystemOfConstraintsAssemblyModel] Number of deviation symbols (nD): 4
INFO:root:[Type: SystemOfConstraintsAssemblyModel] Number of gap symbols (nG): 6
INFO:root:[Type: SystemOfConstraintsAssemblyModel] Generating matrix representation for constraints.
INFO:root:[Type: SystemOfConstraintsAssemblyModel] Matrix representation generated.
INFO:root:[Type: SystemOfConstraintsAssemblyModel] Matrix representation obtained.


4 [v_d_1, w_d_1, beta_d_1, gamma_d_1]


In [9]:
SOCAM.test_zero_deviation_feasibility()

INFO:root:[Type: SystemOfConstraintsAssemblyModel] Invoking the __call__ method.
INFO:root:[Type: SystemOfConstraintsAssemblyModel] The C array representing the coefficients of the linear objective function to be minimized has not been (well) defined
INFO:root:[Type: function:get_gap_symbol_bounds] Getting bounds for gap symbols: [u_g_0, v_g_0, w_g_0, alpha_g_0, beta_g_0, gamma_g_0, s].
DEBUG:root:[Type: function:get_gap_symbol_bounds] Extracted bounds: [[-3, 3], [-3, 3], [-3, 3], [-0.7853981633974483, 0.7853981633974483], [-0.7853981633974483, 0.7853981633974483], [-0.7853981633974483, 0.7853981633974483], [-inf, inf]]
INFO:root:[Type: SystemOfConstraintsAssemblyModel] Returning matrices and bounds for scipy optimization.


{'success': True,
 'status': 0,
 'message': 'Optimization terminated successfully. (HiGHS Status 7: Optimal)',
 'gap_values': array([-0., -0., -0., -0., -0., -0.,  2.]),
 'objective': -2.0}